# Bangladesh: 0197_bootcamp2026_11

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0197_bootcamp2026_11
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'landcover' / '0197_bootcamp2026_11'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""session11_classification.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Landsat - Sentinel Data Analysis in GEE using Python API Session11: Image Classification
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
label = ee.ImageCollection('ESA/WorldCover/v200').filterBounds(roi).first()

def features(img):
    fmask = img.select('Fmask')
    bit0 = fmask.bitwiseAnd(1 << 0).neq(0)
    bit1 = fmask.bitwiseAnd(1 << 1).neq(0)
    bit3 = fmask.bitwiseAnd(1 << 3).neq(0)
    mask = bit0.Or(bit1).Or(bit3)
    ms = img.select('B[2-7]')
    tir = img.select('B10')
    ndvi = ms.normalizedDifference(['B5', 'B4']).rename('ndvi')
    ndwi = ms.normalizedDifference(['B3', 'B5']).rename('ndwi')
    stack = ee.Image.cat([ms, tir, ndvi, ndwi])
    return stack.updateMask(mask.Not())
landsat = ee.ImageCollection('NASA/HLS/HLSL30/v002').filterDate('2025', '2026').filterBounds(roi).filter(ee.Filter.lt('CLOUD_COVERAGE', 40)).map(features)
ms_median = landsat.select('B.*').median()
index_median = landsat.select('n.*').median()
index_std = landsat.select('n.*').reduce(ee.Reducer.stdDev())
dataset = ee.Image.cat([ms_median, index_median, index_std])
dataset
import xarray as xr
label_ds = xr.open_dataset(label, engine='ee', crs='EPSG:3857', geometry=BD_ROI, scale=100)
label_ds = label_ds.squeeze('time').drop_vars('time')
landsat_ds = xr.open_dataset(dataset, engine='ee', crs='EPSG:3857', geometry=BD_ROI, scale=100)
landsat_ds = landsat_ds.squeeze('time').drop_vars('time') * 1
label_ds_int = label_ds.interp(X=landsat_ds.X, Y=landsat_ds.Y, method='nearest')
landsat_aligned, label_aligned = xr.align(landsat_ds, label_ds_int)
label_aligned
landsat_aligned
ds = xr.merge([landsat_aligned, label_aligned])
ds
df = ds.to_dataframe().dropna()
df
x = df.drop('Map', axis=1)
y = df['Map']
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))
df['class_val'] = model.predict(x)
final = df.to_xarray().sortby('Y', 'X')
final['class_val'].plot(x='X', y='Y', cmap='jet')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'landcover', '0197_bootcamp2026_11', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'landcover')  # explicit upload; requires Drive OAuth scope
